# Урок 2. Кодирование текстовой информации

10 класс · I четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [← Урок 1](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-01.ipynb) · [Урок 3 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-03.ipynb)

---

Однобайтные кодировки и таблица ASCII. Unicode, UTF-8 и переменная длина символа. Расчёт объёма текста. Мощность алфавита.

In [ ]:
#@title 🚀 Шаг 1. Регистрация и подготовка урока { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
ФИО = "" #@param {type:"string"}
Класс = "10А" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="10-02", name=ФИО, klass=Класс)

## Разбираемся

### Символ — это число

Компьютер не знает букв. Он хранит числа, а буквы получаются
по **таблице кодировки** — договорённости о том, какое число
какому символу соответствует.

Вся сложность темы в одном: договорённостей было много, и они
несовместимы между собой.

### ASCII: первая договорённость

Стандарт ASCII появился в 1963 году и отвёл под символ **7 бит**,
то есть 128 кодов.

| Коды | Что кодируют |
|---|---|
| 0–31 | управляющие символы (перевод строки, табуляция) |
| 32–47 | пробел и знаки препинания |
| 48–57 | цифры `0`–`9` |
| 65–90 | заглавные латинские `A`–`Z` |
| 97–122 | строчные латинские `a`–`z` |

Три числа стоит запомнить: **48** — код нуля, **65** — код `A`,
**97** — код `a`. Разница между заглавной и строчной ровно 32 —
это не совпадение, а осознанное решение: смена регистра меняет
один-единственный бит.

Восьмой бит остался свободным, и его отдали под расширение —
ещё 128 символов для национальных алфавитов.

### Проблема восьмого бита

Кириллице 128 мест хватало, но каждая страна и каждый производитель
распорядились ими по-своему. Для русского языка возникло сразу
несколько несовместимых кодировок:

| Кодировка | Где применялась |
|---|---|
| KOI8-R | Unix, ранний интернет |
| CP866 | MS-DOS |
| CP1251 | Windows |
| ISO 8859-5 | стандарт ISO |

Одно и то же число означало в них разные буквы. Отсюда легендарные
«кракозябры»: текст, набранный в одной кодировке, открывали в другой
и получали «Ïðèâåò» вместо «Привет».

### Unicode: одна таблица на всё

В 1991 году появился Unicode — единая таблица для **всех** письменностей
мира. Сейчас в ней более 140 тысяч символов: латиница, кириллица,
иероглифы, арабская вязь, математические знаки, эмодзи.

Каждому символу присвоен номер — **кодовая точка**, которую записывают
как `U+041F`. Буква «А» русская — это U+0410, латинская `A` — U+0041.
Это разные символы с разными кодами, хотя выглядят одинаково.

### UTF-8: как Unicode лежит в памяти

Кодовая точка — это номер, а не способ хранения. Хранить все символы
по 4 байта расточительно: английский текст раздулся бы вчетверо.

Решение — **UTF-8**, кодировка с переменной длиной символа:

| Диапазон символов | Байт на символ |
|---|---|
| латиница, цифры (ASCII) | 1 |
| кириллица, греческий, арабский | 2 |
| китайский, японский | 3 |
| эмодзи, редкие письменности | 4 |

Гениальность решения в обратной совместимости: **любой текст ASCII
является корректным UTF-8**. Старые файлы читаются без изменений.

Практическое следствие для расчётов: русский текст в UTF-8 занимает
примерно вдвое больше, чем английский такой же длины.

### Расчёт объёма текста

> Текст из 500 символов кириллицы. Сколько байт займёт файл?

| Кодировка | Байт на символ | Итого |
|---|---|---|
| CP1251 | 1 | 500 байт |
| UTF-8 | 2 | 1000 байт |
| UTF-32 | 4 | 2000 байт |

В задачах ЕГЭ обычно указывают, сколько бит отводится на символ, —
читайте условие внимательно, это ключевой параметр.

## Смотрим, как это работает

### Пример 1. Коды символов

In [ ]:
for символ in "Aa0 Ая":
    print(f"  {символ!r:>5} → код {ord(символ):>6} = U+{ord(символ):04X}")

print()
print("Разница между 'A' и 'a':", ord("a") - ord("A"))
print("Русская 'А' и латинская 'A' — одно и то же?", "А" == "A")

Функция `ord` даёт код символа, `chr` — обратно символ по коду.
Последняя строка показывает частую причину загадочных ошибок:
визуально одинаковые буквы имеют разные коды, и программа честно
считает их разными.

### Пример 2. Шифр Цезаря

Классическое применение кодов символов: сдвиг по алфавиту.

In [ ]:
def цезарь(текст, сдвиг):
    результат = ""
    for символ in текст:
        if "а" <= символ <= "я":
            позиция = ord(символ) - ord("а")
            новая = (позиция + сдвиг) % 32
            результат += chr(ord("а") + новая)
        else:
            результат += символ
    return результат


исходный = "привет мир"
зашифрованный = цезарь(исходный, 3)

print(f"Исходный:      {исходный}")
print(f"Зашифрованный: {зашифрованный}")
print(f"Расшифрованный: {цезарь(зашифрованный, -3)}")

Схема универсальна: переводим букву в позицию в алфавите (вычитая код
первой буквы), сдвигаем с остатком по модулю размера алфавита,
переводим обратно.

Остаток `% 32` обеспечивает зацикливание: после «я» снова идёт «а».
Без него буквы в конце алфавита превратились бы в посторонние символы.

### Пример 3. Сколько байт занимает текст

Проверим теорию экспериментом.

In [ ]:
тексты = ["Hello", "Привет", "你好", "🎉"]

print(f"{'текст':<10} {'символов':>10} {'UTF-8':>8} {'UTF-16':>8} {'UTF-32':>8}")
for текст in тексты:
    print(f"{текст:<10} {len(текст):>10} "
          f"{len(текст.encode('utf-8')):>8} "
          f"{len(текст.encode('utf-16-le')):>8} "
          f"{len(текст.encode('utf-32-le')):>8}")

Метод `.encode()` превращает строку в последовательность байтов.
Видно всё, о чём говорила теория: латиница в UTF-8 занимает по байту,
кириллица по два, иероглифы по три, эмодзи четыре. А UTF-32 честно
тратит четыре байта на всё подряд.

### Пример 4. Что происходит при неверной кодировке

In [ ]:
текст = "Привет"
байты = текст.encode("cp1251")

print(f"Текст:            {текст}")
print(f"Байты в CP1251:   {list(байты)}")
print(f"Читаем как CP1251: {байты.decode('cp1251')}")
print(f"Читаем как KOI8-R: {байты.decode('koi8-r')}")
print(f"Читаем как latin1: {байты.decode('latin1')}")

Байты одни и те же — а текст получается разный. Вот и вся тайна
кракозябр: файл не «испорчен», просто его читают не по той таблице.

## Пробуем сами

### Задача 1. Объём текста

По количеству символов и числу бит на символ верните объём текста
**в байтах**, округлив вверх.

In [ ]:
def объём_байт(символов, бит_на_символ):
    return ...

In [ ]:
si.check("1", объём_байт, [
    ((500, 16), 1000),
    ((500, 8), 500),
    ((100, 5), 63),
    ((8, 1), 1),
])

### Задача 2. Шифр Цезаря

Реализуйте шифр для строчной кириллицы (буквы от «а» до «я», без «ё»).
Символы, не являющиеся буквами, оставляйте без изменений.

Сдвиг может быть отрицательным — тогда получится расшифровка.

In [ ]:
def шифр(текст, сдвиг):
    return ...

In [ ]:
si.check("2", шифр, [
    (("абв", 1), "бвг"),
    (("я", 1), "а"),
    (("привет мир", 3), "тулеих плу"),
    (("абв", 0), "абв"),
])

### Задача 3. Объём в разных кодировках

Русский текст из 1000 символов сохранили в UTF-8. Сколько килобайт
займёт файл? Считайте 1 Кбайт = 1024 байта, ответ округлите
до двух знаков.

Впишите число.

In [ ]:
ответ = 0

si.check_value("3", ответ, "312115dbd13c05f6",
               hint="Кириллица в UTF-8 — два байта на символ.")

## Домашнее задание

### Домашнее задание 1. Смена регистра без готовых методов

Напишите функцию, которая переводит строчные латинские буквы
в заглавные, пользуясь только `ord` и `chr`. Остальные символы
не трогайте.

Вспомните: разница между регистрами ровно 32.

In [ ]:
def в_верхний(текст):
    return ...

In [ ]:
si.check("дз1", в_верхний, [
    ("hello", "HELLO"),
    ("Hello World", "HELLO WORLD"),
    ("abc123", "ABC123"),
    ("", ""),
])

### Домашнее задание 2. Расчёт объёма сообщения

Классическая задача ЕГЭ.

> Каждый символ сообщения кодируется одинаковым количеством бит.
> Сообщение из `символов` символов занимает `байт` байт.
> Сколько символов в алфавите?

Ход решения: переведите объём в биты, разделите на число символов —
получите бит на символ. Мощность алфавита равна 2 в этой степени.

In [ ]:
def мощность_алфавита(символов, байт):
    return ...

In [ ]:
si.check("дз2", мощность_алфавита, [
    ((3072, 1536), 16),
    ((100, 100), 256),
    ((8, 1), 2),
    ((200, 100), 16),
])

### Домашнее задание 3. Определение кодировки по объёму

Напишите функцию, которая по количеству символов и размеру файла
в байтах определяет кодировку: `"ASCII"` (1 байт), `"UTF-16"` (2 байта),
`"UTF-32"` (4 байта). Если не подходит ни одна — верните `"неизвестно"`.

In [ ]:
def определить_кодировку(символов, байт):
    return ...

In [ ]:
si.check("дз3", определить_кодировку, [
    ((100, 100), "ASCII"),
    ((100, 200), "UTF-16"),
    ((100, 400), "UTF-32"),
    ((100, 300), "неизвестно"),
])

---

### Полезно попробовать

Сохраните текстовый файл с русским текстом в разных кодировках
(в блокноте это делается в диалоге сохранения) и сравните размеры.
Разница вдвое между CP1251 и UTF-8 — это ровно то, о чём говорил урок,
только увиденное своими глазами.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 1](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-01.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 3 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-03.ipynb)